In [1]:
import yaml
import os
import pandas as pd
import zipfile
import io
import boto3

In [2]:
os.getcwd()

'/Users/hugolanca/Desktop/Projets/Projet ETL ou ELT/ETL/IDFM/notebooks/ingestion'

In [3]:
s3_client = boto3.client('s3')

In [4]:
file = open('../../config/config.yaml')
yaml_file = yaml.safe_load(file)
file.close()

In [5]:
bucket = yaml_file['S3']['bucket']
path_silver = yaml_file['S3']['chemins_dossiers']['silver']

In [6]:
list = s3_client.list_objects_v2(
Bucket = bucket,
Prefix = path_silver
)

In [7]:
list['Contents']

[{'Key': 'silver/',
  'LastModified': datetime.datetime(2026, 3, 4, 18, 2, 58, tzinfo=tzutc()),
  'ETag': '"d41d8cd98f00b204e9800998ecf8427e"',
  'ChecksumAlgorithm': ['CRC64NVME'],
  'ChecksumType': 'FULL_OBJECT',
  'Size': 0,
  'StorageClass': 'STANDARD'},
 {'Key': 'silver/nb_fer_accessibilite_2024.parquet',
  'LastModified': datetime.datetime(2026, 4, 14, 21, 13, 18, tzinfo=tzutc()),
  'ETag': '"9b79a143cb93bb0bd44317fe298bc7f9"',
  'ChecksumAlgorithm': ['CRC64NVME'],
  'ChecksumType': 'FULL_OBJECT',
  'Size': 2371488,
  'StorageClass': 'STANDARD'}]

In [8]:
key = path_silver + 'nb_fer_accessibilite_2024.parquet'

In [9]:
object = s3_client.get_object(
    Bucket = bucket,
    Key = key
)


In [10]:
df = pd.read_parquet(io.BytesIO(object['Body'].read()))

In [11]:
df

,JOUR,CODE_STIF_TRNS,CODE_STIF_RES,CODE_STIF_ARRET,LIBELLE_ARRET,ID_ZDC,CATEGORIE_TITRE,NB_VALD,zdaid,zdcid,zdaname,zdatown,zdatype,stop_point_id,accessibility_level_id,accessibility_level_name
0,01/01/2024,100,110.0,1.0,PORTE MAILLOT,71379,Amethyste,78.0,415093,71379,Neuilly Porte Maillot,Paris 17e,railStation,415093,3,train accessible sur réservation préalable aup...
1,01/01/2024,100,110.0,1.0,PORTE MAILLOT,71379,Autres titres,885.0,415093,71379,Neuilly Porte Maillot,Paris 17e,railStation,415093,3,train accessible sur réservation préalable aup...
2,01/01/2024,100,110.0,1.0,PORTE MAILLOT,71379,Contrat Solidarité Transport,371.0,415093,71379,Neuilly Porte Maillot,Paris 17e,railStation,415093,3,train accessible sur réservation préalable aup...
3,01/01/2024,100,110.0,1.0,PORTE MAILLOT,71379,Forfait Navigo,1551.0,415093,71379,Neuilly Porte Maillot,Paris 17e,railStation,415093,3,train accessible sur réservation préalable aup...
4,01/01/2024,100,110.0,1.0,PORTE MAILLOT,71379,Forfaits courts,107.0,415093,71379,Neuilly Porte Maillot,Paris 17e,railStation,415093,3,train accessible sur réservation préalable aup...
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
976217,2024-10-03,800,851,560,MONTIGNY SUR L,61327,Amethyste,3.0,47125,61327,Montigny-sur-Loing,Montigny-sur-Loing,railStation,47125,1,gare ou arrêt non accessible
976218,2024-10-03,800,851,560,MONTIGNY SUR L,61327,Contrat Solidarité Transport,8.0,47125,61327,Montigny-sur-Loing,Montigny-sur-Loing,railStation,47125,1,gare ou arrêt non accessible
976219,2024-10-03,800,851,560,MONTIGNY SUR L,61327,Forfait Navigo,118.0,47125,61327,Montigny-sur-Loing,Montigny-sur-Loing,railStation,47125,1,gare ou arrêt non accessible
976220,2024-10-03,800,851,560,MONTIGNY SUR L,61327,Forfaits courts,1.0,47125,61327,Montigny-sur-Loing,Montigny-sur-Loing,railStation,47125,1,gare ou arrêt non accessible


In [12]:
df.shape

(976222, 16)

In [13]:
df.dtypes

JOUR                         object
CODE_STIF_TRNS                int64
CODE_STIF_RES                object
CODE_STIF_ARRET              object
LIBELLE_ARRET                object
ID_ZDC                        Int64
CATEGORIE_TITRE              object
NB_VALD                     float64
zdaid                        object
zdcid                         Int64
zdaname                      object
zdatown                      object
zdatype                      object
stop_point_id                object
accessibility_level_id        Int64
accessibility_level_name     object
dtype: object

## Dédupliquer sur ID_ZDC + JOUR + CATEGORIE_TITRE → pour calculer total_validations sans doublon

In [14]:
df_wduplicate = df.drop_duplicates(subset=['ID_ZDC','JOUR','CATEGORIE_TITRE'])

## Grouper df_validations sur ID_ZDC et sommer NB_VALD pour obtenir total_validations.

In [15]:
df_grouped = df_wduplicate.groupby('ID_ZDC')['NB_VALD'].sum()

In [16]:
df_grouped = df_grouped.reset_index()

## idxmin() sur accessibility_level_id groupé par ID_ZDC

In [17]:
idx_min_pmr = df.groupby('ID_ZDC')['accessibility_level_id'].idxmin()

## idx_min_pmr avec .loc[] sur df pour extraire les lignes correspondantes. 

In [18]:
df_pmr = df.loc[idx_min_pmr.tolist(),['ID_ZDC', 'accessibility_level_id', 'accessibility_level_name','zdaname', 'zdatown', 'zdatype']]

In [19]:
df_pmr = df_pmr.reset_index(drop=True)

In [20]:
df_gold = df_pmr.merge(
    df_grouped,
    on = 'ID_ZDC',
    how = 'left'
)

In [21]:
df_gold

,ID_ZDC,accessibility_level_id,accessibility_level_name,zdaname,zdatown,zdatype,NB_VALD
0,59403,1,gare ou arrêt non accessible,Angerville,Angerville,railStation,35070.0
1,59420,1,gare ou arrêt non accessible,Boigneville,Boigneville,railStation,11021.0
2,59429,1,gare ou arrêt non accessible,Monnerville,Monnerville,railStation,6047.0
3,59447,1,gare ou arrêt non accessible,Buno - Gironville,Buno-Bonnevaux,railStation,11456.0
4,59450,1,gare ou arrêt non accessible,Guillerval,Guillerval,railStation,3463.0
...,...,...,...,...,...,...,...
407,424296,1,gare ou arrêt non accessible,Gare de Villiers Neauphle Pontchartrain,Villiers-Saint-Frédéric,railStation,200109.0
408,424396,3,train accessible sur réservation préalable aup...,Ivry-sur-Seine,Ivry-sur-Seine,railStation,434027.0
409,426280,1,gare ou arrêt non accessible,Rosny Bois Perrier,Rosny-sous-Bois,railStation,1768561.0
410,427230,4,train accessible sur demande auprès d'un agent...,Robinson,Sceaux,railStation,1437630.0


In [22]:
df_gold = df_gold.sort_values(by=['accessibility_level_id', 'NB_VALD'], ascending=[True, False]).reset_index(drop=True)

In [23]:
df_gold['rank_priorite'] = df_gold.index+1

In [24]:
df_gold

,ID_ZDC,accessibility_level_id,accessibility_level_name,zdaname,zdatown,zdatype,NB_VALD,rank_priorite
0,71718,1,gare ou arrêt non accessible,Val de Fontenay,Fontenay-sous-Bois,railStation,5829092.0,1
1,71607,1,gare ou arrêt non accessible,Paris-Bercy Bourgogne - Pays d'Auvergne,Paris 12e,railStation,4500239.0,2
2,61926,1,gare ou arrêt non accessible,Melun,Melun,railStation,4292702.0,3
3,73620,1,gare ou arrêt non accessible,Saint-Michel Notre-Dame,Paris 5e,railStation,4143502.0,4
4,69568,1,gare ou arrêt non accessible,Villeneuve-Saint-Georges,Villeneuve-Saint-Georges,railStation,3592702.0,5
...,...,...,...,...,...,...,...,...
407,65227,6,véhicule accessible en toute autonomie,Épinay - Villetaneuse,Épinay-sur-Seine,railStation,376122.0,408
408,65749,6,véhicule accessible en toute autonomie,Groslay,Groslay,railStation,347344.0,409
409,66795,6,véhicule accessible en toute autonomie,Bouffémont - Moisselles,Bouffémont,railStation,239730.0,410
410,65910,6,véhicule accessible en toute autonomie,Ermont Halte,Ermont,railStation,199528.0,411
